# 02. Preprocessing Encoding

In [ ]:
import sys
import warnings
import json
from pathlib import Path

# 1. Tự động tìm PROJECT_ROOT chứa folder 'src'
current_path = Path.cwd().resolve()
PROJECT_ROOT = current_path
for parent in [current_path] + list(current_path.parents):
    if (parent / 'src').is_dir():
        PROJECT_ROOT = parent
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

# Import các helper từ src/
from src.data.load_data import load_abalone_data
from src.data.split_data import split_features_target, attach_target
from src.features.feature_engineering import (
    prepare_abalone_feature_groups,
    build_encoded_preprocessor,
    build_standard_scaled_preprocessor,
    transform_with_preprocessor
)

warnings.filterwarnings('ignore')

In [ ]:
# Tải dữ liệu từ folder gốc
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'abalone.csv'
df = load_abalone_data(DATA_PATH)

# Xác định các nhóm đặc trưng
feature_groups = prepare_abalone_feature_groups()
target_col = feature_groups['target_col']
categorical_cols = feature_groups['categorical_cols']
numeric_cols = feature_groups['numeric_cols']

# Chia dữ liệu theo tỷ lệ 70/30
X_train, X_test, y_train, y_test = split_features_target(
    df, target_col, test_size=0.3, random_state=42
)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Test: {X_test.shape}")

In [ ]:
# Chỉ mã hóa biến phân loại Sex
encoded_preprocessor = build_encoded_preprocessor(categorical_cols)
X_train_encoded, X_test_encoded = transform_with_preprocessor(encoded_preprocessor, X_train, X_test)

print("Preview dữ liệu chỉ mã hóa:")
display(X_train_encoded.head(3))

In [ ]:
# Mã hóa Sex + Chuẩn hóa các biến số (StandardScaler)
standard_preprocessor = build_standard_scaled_preprocessor(categorical_cols, numeric_cols)
X_train_standard, X_test_standard = transform_with_preprocessor(standard_preprocessor, X_train, X_test)

print("Preview dữ liệu đã chuẩn hóa (Standard Scale):")
display(X_train_standard.head(3))

In [ ]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

# Ghép lại với biến mục tiêu và lưu file
attach_target(X_train_encoded, y_train).to_csv(processed_dir / 'abalone_train_encoded.csv', index=False)
attach_target(X_test_encoded, y_test).to_csv(processed_dir / 'abalone_test_encoded.csv', index=False)
attach_target(X_train_standard, y_train).to_csv(processed_dir / 'abalone_train_standard.csv', index=False)
attach_target(X_test_standard, y_test).to_csv(processed_dir / 'abalone_test_standard.csv', index=False)

print(f"Đã lưu 4 file dữ liệu vào: {processed_dir}")